# Monday Morning at NorthStar

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023.*

**Estimated time: 10–15 minutes. Run every cell top to bottom.**

This is your first hands-on notebook. Run it **before class** — the goal is for you to *feel* what ML can do first, and *understand why* later.

## The Scene

It is 9:12 AM on a Monday in January 2023. You are **Sarah Chen**, starting your second week as Customer Experience Analyst at NorthStar Retail — a mid-sized online clothing store.

**Aisha Patel** from Customer Service walks into your cube holding a USB drive:

> *"Priya wants sentiment on every customer review we got in December. All ten thousand of them. By Friday. Apparently you're the new analyst so — good luck."*

She drops the drive on your desk and leaves.

You open the CSV. 10,000 rows. Reading one review takes about 15 seconds. That's **42 hours of reading** — more than a work-week — before you write a single line of analysis.

You could try a shortcut: write `if-else` rules that look for positive and negative words. Or you could try this "Machine Learning" thing you've been hearing about.

Let's try both.

## Step 1 — Set up

We'll use a lightweight sentiment model called **TextBlob**. It is a small library (no large model downloads, no internet required after install) that can classify a piece of English text as positive or negative with one line of code.

In [3]:
# Run this cell first. It installs TextBlob if needed, then loads it.
# Nothing to edit — just click this cell and press Shift+Enter.
try:
    from textblob import TextBlob
    from textblob.sentiments import NaiveBayesAnalyzer
except ImportError:
    print("TextBlob not found — installing now (one-time, ~10 seconds)...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "textblob"])
    from textblob import TextBlob
    from textblob.sentiments import NaiveBayesAnalyzer
    print("✅ Installed.")

# Download the NLTK movie-reviews corpus that NaiveBayesAnalyzer was trained on.
# First run only — cached afterwards.
import nltk
nltk.download("movie_reviews", quiet=True)
nltk.download("punkt_tab", quiet=True)

import pandas as pd
print("✅ Libraries ready.")

✅ Libraries ready.


## Step 2 — The reviews

Here are 5 real-looking reviews from NorthStar's December haul. Read them carefully before running the next cell — try to guess each one's sentiment in your head.

In [4]:
reviews = [
    "Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.",
    "The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed.",
    "Great value for the price. Not amazing but not bad — I'd order again for basics.",
    "I ordered size M as usual and it's enormous. Runs at least two sizes large. Returning.",
    "Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.",
]

for i, r in enumerate(reviews, 1):
    print(f'{i}. {r}\n')

1. Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.

2. The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed.

3. Great value for the price. Not amazing but not bad — I'd order again for basics.

4. I ordered size M as usual and it's enormous. Runs at least two sizes large. Returning.

5. Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.



### Pause & Predict

Before running the next cell, predict in your head:
- Which of the 5 reviews is **positive**?
- Which is **negative**?
- Is any one of them **tricky**?

Write your predictions down (even just mentally) before scrolling.

## Step 3 — Attempt 1: hand-written rules

Let's try the traditional approach. We'll write a rule: if a review contains more positive words than negative words, label it positive.

This is how programmers solved problems before ML. Watch what happens.

In [5]:
POSITIVE_WORDS = ['love', 'great', 'amazing', 'good', 'perfect', 'premium']
NEGATIVE_WORDS = ['bad', 'disappointed', 'late', 'enormous', 'terrible', 'worst']

def rule_based_sentiment(text):
    text_lower = text.lower()
    pos = sum(1 for w in POSITIVE_WORDS if w in text_lower)
    neg = sum(1 for w in NEGATIVE_WORDS if w in text_lower)
    if pos > neg:
        return 'positive'
    elif neg > pos:
        return 'negative'
    else:
        return 'neutral / unclear'

for i, r in enumerate(reviews, 1):
    label = rule_based_sentiment(r)
    print(f'{i}. [{label:17}]  {r[:70]}...')

1. [positive         ]  Absolutely love this jacket! Fits perfectly and the fabric feels premi...
2. [negative         ]  The shirt arrived two weeks late and the color is nothing like the pho...
3. [positive         ]  Great value for the price. Not amazing but not bad — I'd order again f...
4. [negative         ]  I ordered size M as usual and it's enormous. Runs at least two sizes l...
5. [neutral / unclear]  Sure, it's 'fine', if you enjoy paying $85 for something that comes ap...


### What do you notice?

Look carefully at what the rule-based classifier labelled each review. Compare to your own predictions.

- Review 3 ("Great value... not amazing but not bad") — does the rule handle mixed opinions well?
- Review 5 ("Sure, it's 'fine', if you enjoy paying $85...") — this is sarcasm. Did the rule catch it?
- What other words would you need to add to the lists? When would this ever end?

**This is the core problem the rule-based approach was never going to solve.** You cannot write a finite rulebook for language. People are too creative.

## Step 4 — Attempt 2: Machine Learning

Now let's try ML. TextBlob includes a **Naive Bayes classifier** that was trained on ~2,000 labelled movie reviews from the NLTK corpus. It learned which words and phrases tend to appear in positive vs. negative text — we did not write those rules ourselves.

We load that classifier explicitly and use it with one line.

**No hand-written rules. One line.**

In [6]:
# Instantiate once — NaiveBayesAnalyzer loads the NLTK classifier on creation,
# so creating it inside the loop would reload it on every call.
_analyzer = NaiveBayesAnalyzer()

def ml_sentiment(text):
    blob = TextBlob(text, analyzer=_analyzer)
    res = blob.sentiment  # Sentiment(classification='pos'/'neg', p_pos=..., p_neg=...)
    confidence = res.p_pos if res.classification == "pos" else res.p_neg

    # If the model is close to 50/50, call it neutral rather than forcing a label.
    if 0.45 <= res.p_pos <= 0.55:
        return "neutral", confidence

    label = "positive" if res.classification == "pos" else "negative"
    return label, confidence

print(f"{'#':3} {'label':10} {'confidence':>10}  review")
print('-' * 90)
for i, r in enumerate(reviews, 1):
    label, confidence = ml_sentiment(r)
    print(f'{i:<3} {label:10} {confidence:>10.2%}  {r[:60]}...')

#   label      confidence  review
------------------------------------------------------------------------------------------
1   positive       99.84%  Absolutely love this jacket! Fits perfectly and the fabric f...
2   positive       91.50%  The shirt arrived two weeks late and the color is nothing li...
3   negative       55.32%  Great value for the price. Not amazing but not bad — I'd ord...
4   positive       88.36%  I ordered size M as usual and it's enormous. Runs at least t...
5   neutral        54.17%  Sure, it's 'fine', if you enjoy paying $85 for something tha...


### What do you notice?

Compare the ML output to your own predictions, and to the rule-based classifier.

- The **confidence score** (between 0% and 100%) tells you *how sure* the classifier is — not just *which label* it picked. Lower confidence = a signal to route a case to a human reviewer.
- **Look closely at review 5** (the sarcastic one: *"Sure, it's 'fine', if you enjoy paying $85..."*). What did the model label it? Is that right?
- **Look at review 4** (the sizing complaint). The word 'enormous' isn't negative by itself — the model may have been unsure.
- Notice that you did not have to teach the model about *every* word. It generalised from the labelled examples it saw during training.

**Key insight:** The ML model is better than the rules *on average*, but it **still makes mistakes** — especially on sarcasm, negation, and context-heavy text. It is a trained Naive Bayes classifier, not magic. Keep that in mind for the next step.

### 🔍 How does TextBlob actually decide? (the two analyzers)

*- [Opus 4.8] — explainer cell added by the assistant; not part of the original notebook.*

A natural question: *does TextBlob look our text up in some database of reviews?* **No.** Nothing
matches the whole sentence against stored examples. Both of TextBlob's engines work at the
**word level** — and this notebook uses **both**:

| | **NaiveBayesAnalyzer** (Step 4 above) | **PatternAnalyzer** (Step 5 below, the default) |
|---|---|---|
| Type | A trained ML classifier | A lexicon / rule-based scorer |
| Built-in data | A model **learned from 2,000 labeled movie reviews** (the `movie_reviews` corpus we downloaded in Step 1) | A dictionary of **words → polarity scores** baked into the library |
| Output | `classification` + `p_pos` / `p_neg` | `polarity` from −1 to +1 |
| Cost | **~3 s one-time training**, then ~0.2 ms/review | no training; ~0.1 ms/review |

**NaiveBayesAnalyzer** (what `ml_sentiment` uses). Before it can predict anything it must **train**:
read all 2,000 movie reviews and count how often each word appears in positive vs negative ones,
learning probabilities like `P("love" | positive)` = high. To then classify *our* text it splits it
into a "bag of words" (order ignored), looks up those learned probabilities, and combines them with
**Bayes' theorem**:

```
P(positive | text) ∝ P(positive) × P(w₁|pos) × P(w₂|pos) × …
```

Whichever side scores higher wins. It's called *naive* because it pretends every word is
independent. The 2,000 reviews themselves are **not** consulted at prediction time — only the
probabilities learned from them.

**PatternAnalyzer** (what `fast_sentiment` uses next). It has a built-in dictionary where words
carry scores (`love` ≈ +0.5, `terrible` ≈ −1.0), plus rules for negation (`not good` flips the
sign) and intensifiers (`very good` boosts it). It scores each word and **averages** them into one
polarity. No model, no training — just fast lookups.

**Why Step 5 switches to PatternAnalyzer.** It is *not* that NaiveBayes predicts slowly — once
trained it's about as fast as the lexicon (~0.2 ms/review), and reused properly it would label all
10,000 reviews in a couple of seconds. The real issues are (1) its **one-time ~3 s training**, and
(2) a nasty trap: if you create a new `NaiveBayesAnalyzer()` *inside* the loop it **retrains on
every review** (~3 s each) → 10,000 reviews ≈ **8 hours**. That re-instantiation is exactly the bug
Step 4 avoids by building `_analyzer` **once**, outside the loop. PatternAnalyzer needs no model at
all, so there's nothing to train and nothing to accidentally repeat.

### 🔍 Do all trained models just look words up in a dictionary?
*- [Opus 4.8] — explainer cell added by the assistant; not part of the original notebook.*

Short answer: **no.** Only *lexicon* methods look up predefined scores. Most **trained** models
store learned *numbers* and **compute** a fresh answer every time — they never stored the answer.

**Lookup vs. compute:**

| | What it stores | How it answers |
|---|---|---|
| **Dictionary / lexicon** (PatternAnalyzer) | word → predefined score | look the word up, return the stored value |
| **Trained model** (NaiveBayes, neural nets, …) | learned **parameters** (weights, probabilities) | **calculate** a result from them — the answer was never stored |

- **PatternAnalyzer** really is a dictionary (`love → +0.5`) — but it's hand-written rules, not "trained".
- **NaiveBayesAnalyzer** is trained: it looks up each word's learned probability, then **combines
  them with Bayes' theorem** to compute a new result. Ingredients are looked up; the prediction is computed.
- **The mini neural network** (notebook `05_knn_and_mini_nn.ipynb`) has *no* dictionary at all — just
  weight matrices it multiplies.
- **KNN** is the most "lookup-like" (it stores all the training data), yet it still **computes distances**
  to pick neighbors — which is why it's called a *lazy, non-parametric* learner.

One more: many NLP models have a **vocabulary** (word → ID or → vector). That's a dictionary for
turning *text into numbers* — the input doorway — **not** a table of final answers.

**Bottom line:** "trained" means *learned parameters for a computation*, not *memorized a
dictionary of results*.

## Step 5 — The punch line

Now let's run the ML model on **all 10,000 reviews**, not just 5. We'll use a simulated CSV that represents Sarah's actual workload.

In [11]:
# Simulate 10k reviews by repeating and varying our 5 seeds
import random
random.seed(42)

variants = [
    '',
    ' Arrived in good condition.',
    ' Shipping was fine.',
    ' Would recommend to a friend.',
    ' Not what I expected.',
    ' Thanks for the fast delivery!',
    ' The packaging could be better.',
]

big_batch = [random.choice(reviews) + random.choice(variants) for _ in range(10_000)]
print(f'Total reviews to classify: {len(big_batch):,}')

# print big_batch values
print(big_batch)


Total reviews to classify: 10,000
['Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.', "Great value for the price. Not amazing but not bad — I'd order again for basics. Arrived in good condition.", 'The shirt arrived two weeks late and the color is nothing like the photo. Very disappointed. Arrived in good condition.', 'Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again. Thanks for the fast delivery!', "Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash.", "Sure, it's 'fine', if you enjoy paying $85 for something that comes apart after one wash. Would recommend to a friend.", 'Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again.', 'Absolutely love this jacket! Fits perfectly and the fabric feels premium. Will buy again. Arrived in good condition.', 'The shirt arrived two weeks late and the color is nothing like the photo. Very disappo

In [8]:
import time

# NaiveBayesAnalyzer (~0.5–2 s/review) is too slow for 10k reviews.
# PatternAnalyzer is TextBlob's default: lexicon-based, runs in microseconds.
# - [Opus 4.8] Measured 2026-06-14 (dsai-m3 env): the ~0.5–2 s above is NOT a per-
# - [Opus 4.8] prediction cost. A shared, already-trained analyzer does ~0.2 ms/review
# - [Opus 4.8] (10k ≈ 2.3 s) — about as fast as PatternAnalyzer. The real costs are the
# - [Opus 4.8] ~3 s one-time training, and ~2.8 s/review ONLY if a new NaiveBayesAnalyzer
# - [Opus 4.8] is created inside the loop (that re-instantiation is the true cause of the
# - [Opus 4.8] multi-hour hang). Reused properly, NaiveBayes would handle 10k in ~2.3 s.
def fast_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity  # PatternAnalyzer, no model load
    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    else:
        return "neutral"

start = time.time()
labels = [fast_sentiment(r) for r in big_batch]
elapsed = time.time() - start

print(f'Classified {len(big_batch):,} reviews in {elapsed:.1f} seconds.')
print()

result = pd.Series(labels).value_counts()
print('Breakdown:')
print(result.to_string())

Classified 10,000 reviews in 3.7 seconds.

Breakdown:
positive    5873
neutral     2111
negative    2016


### What do you notice?

- **Manual effort for this task:** ~42 hours of reading.
- **ML effort for this task:** what you just saw — a few seconds.
- Sarah's Friday deadline just became trivial. The rest of the week she can spend on *what this report means*, not on reading.

This is why ML matters in business: **scale that humans cannot match, on problems where a human expert would do it if only they had the time.**

## Step 6 — Sarah's moment of doubt

Sarah writes up her findings and walks into Priya's office with the numbers she just saw.

> *Sarah: "Roughly 60% of December reviews are positive, 20% negative, 20% neutral."*

Priya reads the numbers. Then looks up:

> *Priya: "Sarah — are the positive ones **actually** positive? And how do we know the model isn't miscounting? You mentioned earlier it got a sarcastic review wrong. How many more of those are in the 10,000?"*

Sarah opens her mouth and closes it again. She doesn't know.

She can see the model ran. She can see the numbers. But she has no way to *prove* the numbers are right. The sarcastic review from Step 4 is just one error she caught by hand. How many silent errors are hiding in the other 9,995 reviews?

That question — *how sure are we?* — is the single most important question in applied ML. It is the question Lesson 2 will teach you to answer. And Lessons 3 and 4. And every lesson after that, in some form.

**For now, bring Priya's question to class.** The instructor will help you start to answer it.

## ✅ End of Notebook

You have just completed the first step of your pre-class work.

**Next step →** Go back to [`pre-class.md`](../pre-class.md) and continue with Step 2 (Reflect). The whole pre-class cycle takes ~75 minutes. You've used about 15.

When you get to class, Part 1 will revisit what you just saw — but with a real, heavyweight model (Hugging Face transformers) and a harder set of reviews.